# TabPFN parameters, for a gradient-boosting practitioner

With a gradient booster you spend real effort searching hyperparameters: tree depth, learning rate,
number of trees, subsample, column sample, regularization. With TabPFN that search is gone. The
model is **pretrained**, its weights are fixed, and there is no train-time fit to tune. What is left
is a short list of **inference settings**, and the defaults are strong.

This notebook tours them, measures the few that actually move the needle, and shows how to inspect
what TabPFN decided about your data. Everything runs on a small synthetic task so it stays
self-contained.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
plt.style.use('fivethirtyeight')
from sklearn.metrics import roc_auc_score, average_precision_score
from tabpfn import TabPFNClassifier

def make_data(n=6000, n_features=12, noise=1.5, positive_rate=None, seed=0):
    """A synthetic binary task with a nonlinear signal plus noise."""
    rng = np.random.RandomState(seed)
    X = rng.randn(n, n_features)
    signal = X[:, 0] * X[:, 1] + X[:, 2] ** 2 - X[:, 3] * X[:, 4] + 0.8 * X[:, 5]
    logit = signal + noise * rng.randn(n)
    if positive_rate is None:
        label = (logit > np.median(logit)).astype(int)               # balanced
    else:
        label = (logit > np.quantile(logit, 1 - positive_rate)).astype(int)
    table = pd.DataFrame(X, columns=[f"x{i}" for i in range(n_features)])
    return table, label

def expected_calibration_error(y_true, p, bins=15):
    """Average gap between predicted confidence and actual frequency, over equal-width bins."""
    bucket = np.clip(np.digitize(p, np.linspace(0, 1, bins + 1)) - 1, 0, bins - 1)
    error = 0.0
    for b in range(bins):
        in_bin = bucket == b
        if in_bin.any():
            error += in_bin.mean() * abs(y_true[in_bin].mean() - p[in_bin].mean())
    return error

## The map

| Parameter (default) | What it controls | When to touch it |
|---|---|---|
| `n_estimators` (8) | how many differently-preprocessed copies of the data are ensembled | lower for speed, higher for a small accuracy bump |
| `categorical_features_indices` (None) | which columns are treated as categorical | when auto-detection mis-types a column |
| `ignore_pretraining_limits` (False) | allow data beyond the recommended size | many rows or features (expect some degradation) |
| `balance_probabilities` (False) | reweight outputs toward balanced classes | imbalanced data, to move the 0.5 operating point |
| `softmax_temperature` (0.9) | sharpen or soften the predicted probabilities | calibration |
| `inference_config` (None) | override the internal preprocessing | advanced: scaling, outlier clip, categorical thresholds |
| `device`, `inference_precision`, `memory_saving_mode`, `fit_mode` | hardware and memory | performance |
| `model_path` (`'auto'`) | which checkpoint to load | `'auto'` loads v3; rarely changed |
| `tuning_config` (None) | optional post-hoc tuning search | when you want an AutoML-style sweep |
| `random_state` (0) | reproducibility | set it for reproducible runs |

The next cells measure the three a practitioner actually reaches for, then inspect the rest.

## `n_estimators`: the accuracy-vs-speed dial

Each estimator is a differently-preprocessed copy of the data, and TabPFN averages over them. More
estimators reduce the variance of the prediction, at a **linear** compute cost: eight estimators
take about eight times as long to predict as one. The accuracy gain saturates quickly, and exactly
how quickly is dataset-dependent, so there is no universal number. The default of 8 is a reasonable
middle: drop it to 1 or 2 when you need speed, raise it for a marginal, diminishing gain.

## What the calibration knobs do to the output

`balance_probabilities` and `softmax_temperature` reshape the predicted probabilities *without
touching the ranking*. That part is not an empirical claim: both are monotone transforms of the
probability, and a monotone transform cannot change which examples score higher, so AUC and AUPRC
are unchanged by construction. What they change is the *shape* of the output distribution, and with
it the calibration and any fixed-threshold decision. The plots below show that directly, on a
deliberately hard task where the model is uncertain enough for the effect to be visible.

In [ ]:
# balance_probabilities on a HARD, imbalanced task (uncertain model, so the shift is visible).
rng = np.random.RandomState(1)
m = 6000
Xb = rng.randn(m, 4)
logit = 0.5 * Xb[:, 0] - 2.2                                 # weak signal + low base rate (uncertain, ~11% positive)
yb = (rng.rand(m) < 1 / (1 + np.exp(-logit))).astype(int)
Xtr, Xte = pd.DataFrame(Xb[:m // 2]), pd.DataFrame(Xb[m // 2:]); ytr, yte = yb[:m // 2], yb[m // 2:]

p_off = TabPFNClassifier(random_state=0).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
p_on  = TabPFNClassifier(balance_probabilities=True, random_state=0).fit(Xtr, ytr).predict_proba(Xte)[:, 1]

plt.figure(figsize=(6, 3))
plt.hist(p_off, bins=30, alpha=0.6, label="balance_probabilities=False")
plt.hist(p_on,  bins=30, alpha=0.6, label="balance_probabilities=True")
plt.legend(); plt.xlabel("predicted P(class 1)"); plt.title("balance_probabilities reshapes the output")
plt.show()

In [ ]:
# softmax_temperature on a HARD, balanced task (uncertain model, so sharpen/soften shows).
rng = np.random.RandomState(2)
m = 6000
Xt = rng.randn(m, 6)
logit = 0.6 * Xt[:, 0] + 0.4 * Xt[:, 1]
yt = (rng.rand(m) < 1 / (1 + np.exp(-logit))).astype(int)
Xtr, Xte = pd.DataFrame(Xt[:m // 2]), pd.DataFrame(Xt[m // 2:]); ytr, yte = yt[:m // 2], yt[m // 2:]

plt.figure(figsize=(6, 3))
for t in [0.5, 0.9, 2.0]:
    p = TabPFNClassifier(softmax_temperature=t, random_state=0).fit(Xtr, ytr).predict_proba(Xte)[:, 1]
    plt.hist(p, bins=30, alpha=0.5, label=f"temperature={t}")
plt.legend(); plt.xlabel("predicted P(class 1)"); plt.title("softmax_temperature: <1 more confident (toward 0/1), >1 less confident (toward 0.5)")
plt.show()

`balance_probabilities` pushes mass toward the minority class: it raises recall at a 0.5 threshold and, on imbalanced data, worsens calibration (the probabilities drift away from the true base rate).

`softmax_temperature` controls confidence. Below 1 the model is more confident, probabilities are pushed toward 0 and 1, so the across-example histogram spreads to the extremes. Above 1 it is less confident, probabilities are pulled toward 0.5, so the histogram collapses into a peak at 0.5. (Note: less confident per example gives a *more concentrated* histogram, not a flatter one, which is easy to misread.) The default 0.9 sits near the best-calibrated point.

Neither knob moves a ranking metric, so reach for them only when you threshold at a fixed cutoff or need calibrated probabilities, not to improve AUC. (A booster's `scale_pos_weight` is what people often expect to help ranking; this is not that.)

## Inspect what TabPFN decided

You do not tune the preprocessing, but you can read it. After a fit, `inference_config_` holds the
resolved limits and thresholds, and `inferred_feature_schema_` shows the modality TabPFN assigned to
each column (which is what `categorical_features_indices` would override).

In [ ]:
Xq = pd.DataFrame(np.random.RandomState(0).randn(200, 5), columns=["a", "b", "c", "d", "e"])
yq = (Xq["a"] > 0).astype(int)
model = TabPFNClassifier(random_state=0).fit(Xq, yq)

print("Envelope (hard ceilings):")
for field in ["MAX_NUMBER_OF_SAMPLES", "MAX_NUMBER_OF_FEATURES", "MAX_NUMBER_OF_CLASSES"]:
    print(f"  {field} = {getattr(model.inference_config_, field)}")

print("\nPreprocessing thresholds:")
for field in ["MAX_UNIQUE_FOR_CATEGORICAL_FEATURES", "MIN_UNIQUE_FOR_NUMERICAL_FEATURES",
              "MIN_NUMBER_SAMPLES_FOR_CATEGORICAL_INFERENCE", "OUTLIER_REMOVAL_STD"]:
    print(f"  {field} = {getattr(model.inference_config_, field)}")

print("\nInferred column modality (first five columns):")
for feature in model.inferred_feature_schema_.features[:5]:
    print(f"  {feature.name}: {feature.modality.value}")

The **envelope** is the hard ceiling the checkpoint was built for (a million rows, two thousand
features, a hundred and sixty classes). Separately, `ignore_pretraining_limits` guards a softer
*recommended* operating range, and pushing well past it is where accuracy starts to slip. The
**thresholds** are the rules behind the auto-detection: a string column with 30 or fewer distinct
values is treated as categorical, a numeric column needs at least 4 distinct values to stay numeric,
and category inference needs at least 100 rows. Those are the knobs the scaling and categorical
chapters open up through `inference_config`.

## Takeaways

- TabPFN is pretrained, so there is no hyperparameter *search*. You configure inference, and the
  defaults are strong.
- **`n_estimators`** trades compute for a small, saturating accuracy gain; the default is a sensible
  middle.
- **`balance_probabilities`** and **`softmax_temperature`** never change the ranking (a monotone
  transform cannot); they shift the operating point and the calibration. Use them only if you
  threshold at a fixed cutoff or need calibrated probabilities.
- **`categorical_features_indices`** and **`ignore_pretraining_limits`** tell TabPFN the truth about
  your data: which columns are categorical, and whether you are past its recommended size.
- **`inference_config`** is the door to the internal preprocessing (scaling, outlier clipping,
  categorical thresholds), and the envelope and thresholds above are read straight off it.

The shift from gradient boosting: you are not tuning a model, you are configuring a fixed one, and
most of the time the defaults are the right answer.